In [12]:
import ee
import geemap
import pandas as pd

In [17]:
ee.Initialize(project="sundarbans-climate-analysis")

print("Google Earth Engine initialized successfully!")

Google Earth Engine initialized successfully!


In [18]:
districts = ee.FeatureCollection(
    "FAO/GAUL/2015/level2"
)

In [19]:
bangladesh_districts = districts.filter(
    ee.Filter.eq("ADM0_NAME", "Bangladesh")
)

In [20]:
print("Bangladesh district boundary loaded.")

Bangladesh district boundary loaded.


In [21]:
target_districts = bangladesh_districts.filter(
    ee.Filter.inList(
        "ADM2_NAME",
        [
            "Satkhira",
            "Khulna",
            "Bagerhat"
        ]
    )
)

In [22]:
print(
    "Selected districts:"
)

Selected districts:


In [23]:
print(
    target_districts.aggregate_array(
        "ADM2_NAME"
    ).getInfo()
)

['Bagerhat', 'Khulna', 'Satkhira']


In [24]:
study_area = target_districts.geometry()

In [25]:
print("Study area created successfully.")


Study area created successfully.


In [26]:
Map = geemap.Map()

In [27]:
Map.centerObject(
    study_area,
    8
)

In [28]:
Map.addLayer(
    target_districts.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Satkhira-Khulna-Bagerhat"
)

In [29]:
Map

Map(center=[22.408422854933942, 89.45252190666105], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
## Temperature extraction code

In [30]:
era5 = ee.ImageCollection(
    "ECMWF/ERA5_LAND/MONTHLY_AGGR"
)

In [31]:
print("ERA5-Land monthly dataset loaded.")

ERA5-Land monthly dataset loaded.


In [32]:
temperature_collection = (
    era5
    .filterDate(
        "2000-01-01",
        "2026-01-01"
    )
    .select(
        "temperature_2m"
    )
)

In [33]:
print(
    "Number of monthly images:",
    temperature_collection.size().getInfo()
)

Number of monthly images: 312


In [34]:
start_date = ee.Date("2000-01-01")
end_date = ee.Date("2026-01-01")

In [35]:
number_of_months = (
    end_date
    .difference(
        start_date,
        "month"
    )
)

In [36]:
month_sequence = ee.List.sequence(
    0,
    number_of_months.subtract(1)
)

In [51]:
def calculate_monthly_temperature(month_number):


    current_date = start_date.advance(
        month_number,
        "month"
    )

    next_date = current_date.advance(
        1,
        "month"
    )

    monthly_image = (
        temperature_collection
        .filterDate(
            current_date,
            next_date
        )
        .first()
    )

    mean_temperature_kelvin = (
        monthly_image
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=11132,
            maxPixels=1000000
        )
        .get("temperature_2m")
    )

    mean_temperature_celsius = (
        ee.Number(mean_temperature_kelvin)
        .subtract(273.15)
    )

    return ee.Feature(
        None,
        {
            "Year": current_date.get("year"),
            "Month_Number": current_date.get("month"),
            "Month": current_date.format("MMMM"),
            "Date": current_date.format(
                "YYYY-MM-dd"
            ),
            "Mean_Temperature_C": mean_temperature_celsius
        }
    )

In [52]:
monthly_temperature = ee.FeatureCollection(
    month_sequence.map(
        calculate_monthly_temperature
    )
)

In [53]:
print(
    "Monthly temperature calculation completed."
)

Monthly temperature calculation completed.


In [54]:
print(
    "Number of records:",
    monthly_temperature.size().getInfo()
)

Number of records: 312


In [55]:
## CONVERT EARTH ENGINE DATA TO PANDAS DATAFRAME

In [56]:
data = monthly_temperature.getInfo()

In [58]:
rows = []

for feature in data["features"]:

    properties = feature["properties"]

    rows.append({
        "Year": properties.get("Year"),
        "Month_Number": properties.get("Month_Number"),
        "Month": properties.get("Month"),
        "Date": properties.get("Date"),
        "Mean_Temperature_C": properties.get(
            "Mean_Temperature_C"
        )
    })


In [59]:
df = pd.DataFrame(rows)

In [60]:
df = df.sort_values(
    by="Date"
).reset_index(drop=True)

In [61]:
df["Mean_Temperature_C"] = df[
    "Mean_Temperature_C"
].round(2)

In [62]:
print(df.head(12))

    Year  Month_Number      Month        Date  Mean_Temperature_C
0   2000             1    January  2000-01-01               19.74
1   2000             2   February  2000-02-01               21.17
2   2000             3      March  2000-03-01               26.91
3   2000             4      April  2000-04-01               29.42
4   2000             5        May  2000-05-01               29.24
5   2000             6       June  2000-06-01               28.25
6   2000             7       July  2000-07-01               27.62
7   2000             8     August  2000-08-01               27.72
8   2000             9  September  2000-09-01               27.11
9   2000            10    October  2000-10-01               26.82
10  2000            11   November  2000-11-01               24.12
11  2000            12   December  2000-12-01               19.80


In [63]:
## EXPORT TO EXCEL

In [64]:
output_file = "Sundarbans_Monthly_Temperature_2000_2025.xlsx"

In [65]:
df.to_excel(
    output_file,
    index=False
)

In [66]:
print(
    f"Excel file created successfully: {output_file}"
)

Excel file created successfully: Sundarbans_Monthly_Temperature_2000_2025.xlsx


In [67]:
####

In [68]:
print("Total rows:", len(df))

Total rows: 312


In [69]:
print(
    "Date range:",
    df["Date"].min(),
    "to",
    df["Date"].max()
)

Date range: 2000-01-01 to 2025-12-01


In [70]:
print("\nMissing values:")
print(
    df.isnull().sum()
)


Missing values:
Year                  0
Month_Number          0
Month                 0
Date                  0
Mean_Temperature_C    0
dtype: int64


In [71]:
print("\nLast 12 records:")
print(
    df.tail(12)
)


Last 12 records:
     Year  Month_Number      Month        Date  Mean_Temperature_C
300  2025             1    January  2025-01-01               19.92
301  2025             2   February  2025-02-01               23.80
302  2025             3      March  2025-03-01               27.53
303  2025             4      April  2025-04-01               28.86
304  2025             5        May  2025-05-01               29.16
305  2025             6       June  2025-06-01               28.48
306  2025             7       July  2025-07-01               27.74
307  2025             8     August  2025-08-01               27.81
308  2025             9  September  2025-09-01               28.13
309  2025            10    October  2025-10-01               27.39
310  2025            11   November  2025-11-01               23.75
311  2025            12   December  2025-12-01               20.10
